## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

In [15]:
# basic hello world code


In [16]:
# The imports
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import (
    Agent,
    OpenAIChatCompletionsModel,
    Runner,
    set_default_openai_api,
    set_default_openai_client,
    set_tracing_disabled,
    set_tracing_export_api_key,
    trace,
)

In [17]:
# The usual starting point
load_dotenv(override=True)

True

In [18]:
openrouter_base_url = os.getenv("OPENROUTER_BASE_URL")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
client = AsyncOpenAI(
    api_key=openrouter_api_key,
    base_url=openrouter_base_url,
)

set_default_openai_api("chat_completions")
set_default_openai_client(client, use_for_tracing=False)

# OpenRouter runs inference; export traces to platform.openai.com with a real OpenAI key
openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    set_tracing_export_api_key(openai_api_key)
else:
    set_tracing_disabled(True)

In [19]:
# Make an agent with name, instructions, model
model = OpenAIChatCompletionsModel(model="gpt-4.1-nano", openai_client=client)

In [20]:
agent = Agent(name="Jokester", instructions="You are a joke teller", model=model)

In [21]:
# Run the joke with Runner.run(agent, prompt) then print final_output
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
    print(result.final_output)

Why did the autonomous AI agent go to therapy?

Because it kept looping through its own thoughts!


## Now go and look at the trace

https://platform.openai.com/traces

When using OpenRouter for inference, set `OPENAI_API_KEY` in `.env` so traces can be exported to the OpenAI platform. Without it, tracing is disabled to avoid non-fatal 401 errors from sending an OpenRouter key to OpenAI's trace endpoint.

In [27]:
def fetch_trace(res):
    for attr in ("get_trace_output","get_trace","trace_output","trace"):
        if hasattr(res, attr):
            val = getattr(res, attr)
            if callable(val):
                try:
                    return val()
                except TypeError:
                    return val
            return val
    try:
        import json
        return json.dumps(res.__dict__, default=str, indent=2)
    except Exception:
        return repr(res)

trace_text = fetch_trace(result)
print("Trace (best effort):")
print(trace_text)
print("\nFinal Output:", getattr(result, 'final_output', None))


Trace (best effort):
{
  "input": "Tell a joke about Autonomous AI Agents",
  "new_items": [
    "MessageOutputItem(agent=Agent(name='Jokester', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are a joke teller', prompt=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x109dffe30>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True), raw_item=ResponseOutputMessage(id='__fake_id__', content=[ResponseOutputText(annotations=[], text='Why did the autonom